# Feature Engineering

In [1]:
import pandas as pd

## Load data

In [2]:
df = pd.read_csv('./clean_data_after_eda.csv')
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [3]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


## Feature engineering

### Difference between off-peak prices in December and preceding January


In [4]:
price_df = pd.read_csv('price_data.csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [5]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


In [6]:
df = pd.merge(df, diff, on='id', how='left')
df[['id', 'offpeak_diff_dec_january_energy', 'offpeak_diff_dec_january_power']].head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,24011ae4ebbe3035111d65fa7c15bc57,0.020057,3.700961
1,d29c2c54acc38ff3c0614d0a653813dd,-0.003767,0.177779
2,764c75f661154dac3a6c254cd082ea7d,-0.004670,0.177779
3,bba03439a292a1e166f80264c16191cb,-0.004547,0.177779
4,149d57cf92fc41cf94415803a877cb4b,-0.006192,0.162916


In [7]:
price_cols = ['price_off_peak_var', 'price_peak_var', 'price_mid_peak_var',
              'price_off_peak_fix', 'price_peak_fix', 'price_mid_peak_fix']
price_cols = [c for c in price_cols if c in price_df.columns]

# Spread (max - min) per id, across all months of the year, for every price component
price_spread = (
    price_df.groupby('id')[price_cols]
    .agg(lambda x: x.max() - x.min())
    .reset_index()
)
price_spread.columns = ['id'] + [f'{c}_spread' for c in price_cols]

# Mean price per id -- the customer's "typical" price level (useful alongside the spread)
price_mean = price_df.groupby('id')[price_cols].mean().reset_index()
price_mean.columns = ['id'] + [f'{c}_mean' for c in price_cols]

df = df.merge(price_spread, on='id', how='left')
df = df.merge(price_mean, on='id', how='left')

df[[c for c in df.columns if 'spread' in c or '_mean' in c]].head()

,price_off_peak_var_spread,price_peak_var_spread,price_mid_peak_var_spread,price_off_peak_fix_spread,price_peak_fix_spread,price_mid_peak_fix_spread,price_off_peak_var_mean,price_peak_var_mean,price_mid_peak_var_mean,price_off_peak_fix_mean,price_peak_fix_mean,price_mid_peak_fix_mean
0,0.028554,0.018480,0.073873,3.700961,24.437330,16.291555,0.124787,0.100749,0.066530,40.942265,22.352010,14.901340
1,0.005334,0.085483,0.000000,0.177780,0.000000,0.000000,0.149609,0.007124,0.000000,44.311375,0.000000,0.000000
2,0.004670,0.001281,0.000000,0.177779,0.000000,0.000000,0.170512,0.088421,0.000000,44.385450,0.000000,0.000000
3,0.004547,0.000000,0.000000,0.177779,0.000000,0.000000,0.151210,0.000000,0.000000,44.400265,0.000000,0.000000
4,0.008161,0.004169,0.003541,0.162916,0.097749,0.065166,0.124174,0.103638,0.072865,40.688156,24.412893,16.275263


### Peak vs. off-peak spread

In [8]:
if 'price_off_peak_var_mean' in df.columns and 'price_peak_var_mean' in df.columns:
    df['peak_offpeak_var_gap'] = df['price_peak_var_mean'] - df['price_off_peak_var_mean']

if 'price_off_peak_fix_mean' in df.columns and 'price_peak_fix_mean' in df.columns:
    df['peak_offpeak_fix_gap'] = df['price_peak_fix_mean'] - df['price_off_peak_fix_mean']


---
### Date-based features

Raw dates aren't directly usable by most models. Turn them into durations that
actually carry meaning for churn: how long has this customer been with us, how
soon does their contract end/renew, how recently was their product changed.

We use `date_end` as the reference "today" since that's roughly when churn is
evaluated for this dataset (3 months after).

In [9]:
date_cols_present = [c for c in ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']
                      if c in df.columns]

if 'date_activ' in df.columns and 'date_end' in df.columns:
    df['tenure_days'] = (df['date_end'] - df['date_activ']).dt.days

if 'date_renewal' in df.columns and 'date_end' in df.columns:
    df['days_to_renewal'] = (df['date_renewal'] - df['date_end']).dt.days

if 'date_modif_prod' in df.columns and 'date_end' in df.columns:
    df['days_since_modif_prod'] = (df['date_end'] - df['date_modif_prod']).dt.days

if 'date_activ' in df.columns:
    df['activ_month'] = df['date_activ'].dt.month
    df['activ_year'] = df['date_activ'].dt.year

# Drop the raw dates 
df = df.drop(columns=date_cols_present)
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_cons_year,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,...,price_off_peak_fix_mean,price_peak_fix_mean,price_mid_peak_fix_mean,peak_offpeak_var_gap,peak_offpeak_fix_gap,tenure_days,days_to_renewal,days_since_modif_prod,activ_month,activ_year
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,0.00,0,0.0,1.78,0.114481,...,40.942265,22.35201,14.90134,-0.024038,-18.590255,1096,-358,227,6,2013
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,189.95,0,0.0,16.27,0.145711,...,44.311375,0.00000,0.00000,-0.142485,-44.311375,2566,-365,2566,8,2009
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,47.96,0,0.0,38.72,0.165794,...,44.385450,0.00000,0.00000,-0.082090,-44.385450,2192,-365,2192,4,2010


### Consumption & forecast based features

- How accurate is PowerCo's own forecast for this customer? A big forecast error
  could itself be a sign of an unstable/unhappy account.
- Normalize last month's consumption against the 12-month average, to see if
  usage is trending down (possible early churn signal) independent of customer size.

In [10]:
if 'cons_12m' in df.columns and 'forecast_cons_12m' in df.columns:
    df['forecast_error_cons_12m'] = df['cons_12m'] - df['forecast_cons_12m']

if 'cons_last_month' in df.columns and 'cons_12m' in df.columns:
    # avoid divide-by-zero
    df['last_month_vs_avg_monthly_cons'] = df['cons_last_month'] / (df['cons_12m'] / 12).replace(0, pd.NA)

if 'net_margin' in df.columns and 'nb_prod_act' in df.columns:
    df['margin_per_active_product'] = df['net_margin'] / df['nb_prod_act'].replace(0, pd.NA)

if 'cons_12m' in df.columns and 'pow_max' in df.columns:
    df['cons_per_subscribed_power'] = df['cons_12m'] / df['pow_max'].replace(0, pd.NA)


### Categorical encoding

`channel_sales` and `origin_up` are hashed categorical codes, one-hot encode them. `has_gas` is already binary-ish, just
standardize it to 0/1.

In [11]:
if 'has_gas' in df.columns:
    df['has_gas'] = df['has_gas'].map({'t': 1, 'f': 0, 'y': 1, 'n': 0, True: 1, False: 0}).fillna(df['has_gas'])

categorical_to_encode = [c for c in ['channel_sales', 'origin_up', 'activity_new'] if c in df.columns]
df = pd.get_dummies(df, columns=categorical_to_encode, dummy_na=True, prefix=categorical_to_encode)

df.shape

(14606, 79)

### Remove low-value columns

Drop anything that's constant and
flag near-duplicate raw price columns now that we've summarized them into
`_mean` / `_spread` features above.

In [12]:
# Constant columns add nothing to a model
constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
print("Dropping constant columns:", constant_cols or "none")
df = df.drop(columns=constant_cols)

df.shape

Dropping constant columns: ['channel_sales_nan', 'origin_up_nan']


(14606, 77)

### Final check & save

In [13]:
print(f"Final shape: {df.shape}")
df.info()

Final shape: (14606, 77)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14606 entries, 0 to 14605
Data columns (total 77 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   id                                              14606 non-null  object 
 1   cons_12m                                        14606 non-null  int64  
 2   cons_gas_12m                                    14606 non-null  int64  
 3   cons_last_month                                 14606 non-null  int64  
 4   forecast_cons_12m                               14606 non-null  float64
 5   forecast_cons_year                              14606 non-null  int64  
 6   forecast_discount_energy                        14606 non-null  float64
 7   forecast_meter_rent_12m                         14606 non-null  float64
 8   forecast_price_energy_off_peak                  14606 non-null  float64
 9   forecast_price

In [14]:
df.to_csv('data_for_predictions.csv', index=False)
print("Saved: data_for_predictions.csv")

Saved: data_for_predictions.csv
